### 1251. Average Selling Price

* Write a solution to find the average selling price for each product. average_price should be rounded to 2 decimal places. If a product does not have any sold units, its average selling price is assumed to be 0.

In [0]:
%sql 
CREATE OR REPLACE TABLE Prices (
  product_id INT,
  start_date DATE,
  end_date DATE,
  price INT,
  PRIMARY KEY (product_id, start_date, end_date)
);

CREATE OR REPLACE TABLE UnitsSold(
  product_id INT,
  purchase_date DATE,
  units INT
);

INSERT INTO Prices (product_id, start_date, end_date, price) VALUES 
(1, '2019-02-17', '2019-02-28', 5),
(1, '2019-03-01', '2019-03-22', 20),
(2, '2019-02-01', '2019-02-20', 15),
(2, '2019-02-21', '2019-03-31', 30),
(3, '2019-01-01', '2019-12-31', 50);

INSERT INTO UnitsSold (product_id, purchase_date, units) VALUES
(1, '2019-02-25', 100),
(1, '2019-03-01', 15),
(2, '2019-02-10', 200),
(2, '2019-03-22', 30),
(2, '2019-03-22', 30);

In [0]:
from pyspark.sql import functions as F

df_prices = spark.table("workspace.default.prices")
df_unitsSold = spark.table("workspace.default.unitssold")

df_unitsSold_dedup = df_unitsSold.dropDuplicates()

df_result = df_prices.join(df_unitsSold_dedup,
    (df_prices['product_id'] == df_unitsSold_dedup['product_id']) & 
    (df_unitsSold_dedup['purchase_date'] >= df_prices["start_date"]) & 
    (df_unitsSold_dedup['purchase_date'] <= df_prices["end_date"])                      
    , "left") \
    .groupBy(df_prices["product_id"]).agg(
        F.round(
            F.coalesce(
                F.sum(
                    df_unitsSold_dedup['units'] * df_prices['price']
                ) / F.when(F.sum(df_unitsSold_dedup['units']) != 0, F.sum(df_unitsSold_dedup['units'])).otherwise(F.lit(None))
            , F.lit(0))
        , 2).alias("average_price")
    )

display(df_result)

In [0]:
%sql

WITH cleaned_UnitsSold AS (
  SELECT DISTINCT product_id, purchase_date, units FROM UnitsSold
) SELECT p.product_id, ROUND(COALESCE(SUM(u.units * p.price) / NULLIF(SUM(u.units), 0), 0), 2) AS average_price
FROM Prices p left join cleaned_UnitsSold u ON p.product_id=u.product_id AND u.purchase_date BETWEEN p.start_date AND p.end_date
GROUP BY p.product_id